## 0. 学生信息与运行环境

In [1]:
import os, uuid, hashlib, platform, getpass

# ===== 1. 填写学号 =====
STUDENT_ID = 20244080205  

# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

# ===== 3. 机器指纹（绑定当前电脑） =====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] base
[学号]     20244080205
[机器指纹] a41d18205864c983
[RUN_ID]   20244080205-a41d18205864c983


---

# 1. 字符串的乐趣

三个函数按题目约定不做错误检查，只处理英文字母、数字和空格。

### 1.1 `is_palindrome`

先统一小写、只保留字母和数字得到 `compact`，再与 `compact[::-1]` 比较，两遍扫描，$O(n)$。

In [3]:
def is_palindrome(s):
    compact = "".join(ch for ch in s.lower() if ch.isalnum())
    return compact == compact[::-1]

In [4]:
palindromes = ["level", "kayak", "pop", "tacocat", "T A C O cat",
               "rats live on no evil star", "Never odd or even",
               "Was it a car or a cat I saw?"]
non_palindromes = ["goat", "python", "abc", "Palindrome"]

for s in palindromes:
    print(f"{s!r:28s} -> {is_palindrome(s)}")
for s in non_palindromes:
    print(f"{s!r:28s} -> {is_palindrome(s)}")

'level'                      -> True
'kayak'                      -> True
'pop'                        -> True
'tacocat'                    -> True
'T A C O cat'                -> True
'rats live on no evil star'  -> True
'Never odd or even'          -> True
'Was it a car or a cat I saw?' -> True
'goat'                       -> False
'python'                     -> False
'abc'                        -> False
'Palindrome'                 -> False


### 1.2 `is_abecedarian`

去掉空格后逐对比较相邻字母，出现一次逆序立即返回 `False`；相等表示字母重复，是允许的。

In [5]:
def is_abecedarian(s):
    letters = "".join(ch.lower() for ch in s if ch.isalpha())
    for i in range(len(letters) - 1):
        if letters[i] > letters[i + 1]:
            return False
    return True

In [6]:
cases = ["adder", "beet", "abcd efgh xyz", "a", "", "aabbcc",
         "dog", "cat", "zyx"]

for s in cases:
    print(f"{s!r:18s} -> {is_abecedarian(s)}")

'adder'            -> True
'beet'             -> True
'abcd efgh xyz'    -> True
'a'                -> True
''                 -> True
'aabbcc'           -> True
'dog'              -> False
'cat'              -> False
'zyx'              -> False


### 1.3 `remove_vowels`

用生成器表达式保留所有非元音字符；全部是元音时返回空串 `''`。

In [7]:
def remove_vowels(s):
    return "".join(ch for ch in s if ch.lower() not in "aeiou")

In [8]:
for s in ["cat", "audio", "aeiou", "xyz", "Programming", ""]:
    print(f"{s!r:14s} -> {remove_vowels(s)!r}")

'cat'          -> 'ct'
'audio'        -> 'd'
'aeiou'        -> ''
'xyz'          -> 'xyz'
'Programming'  -> 'Prgrmmng'
''             -> ''


---

# 2. 列表的乐趣

### 2.1 `list_reverse`

先检查参数是不是 `list`，再用切片 `lst[::-1]` 返回**新**列表，原列表保持不变。

In [9]:
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError(f"参数必须是 list，收到 {type(lst).__name__}")
    return lst[::-1]

In [10]:
print("list_reverse([1, 2, 3]) =", list_reverse([1, 2, 3]))
print("list_reverse([])        =", list_reverse([]))

original = [1, 2, 3]
reversed_copy = list_reverse(original)
print("原列表保持不变：", original, "| 返回新列表：", reversed_copy)

for bad in [(1, 2, 3), "abc", 123]:
    try:
        list_reverse(bad)
    except TypeError as e:
        print(f"{bad!r:12s} -> TypeError: {e}")

list_reverse([1, 2, 3]) = [3, 2, 1]
list_reverse([])        = []
原列表保持不变： [1, 2, 3] | 返回新列表： [3, 2, 1]
(1, 2, 3)    -> TypeError: 参数必须是 list，收到 tuple
'abc'        -> TypeError: 参数必须是 list，收到 str
123          -> TypeError: 参数必须是 list，收到 int


### 2.2 `is_sorted`

只遍历一次：比较相邻两项，一旦发现 `seq[i] > seq[i + 1]` 就提前返回 `False`；循环正常结束说明序列升序。

In [11]:
def is_sorted(seq):
    for i in range(len(seq) - 1):
        if seq[i] > seq[i + 1]:
            return False
    return True

In [12]:
seqs = [[1, 2, 3], [1, 1, 2], [3, 2, 1], [], (1, 2, 3),
        "abc", range(5), [2, 1, 3]]

for seq in seqs:
    print(f"{str(seq):16s} -> {is_sorted(seq)}")

[1, 2, 3]        -> True
[1, 1, 2]        -> True
[3, 2, 1]        -> False
[]               -> True
(1, 2, 3)        -> True
abc              -> True
range(0, 5)      -> True
[2, 1, 3]        -> False


---

# 3. 作为向量的元组

$d$ 维向量用长度为 $d$ 的元组表示。下面四个函数都在入口检查类型，出错时抛出带说明的异常。

### 3.1 `vec_scalar_mult`

In [13]:
def vec_scalar_mult(t, s):
    if not isinstance(t, tuple):
        raise TypeError(f"t 必须是元组，收到 {type(t).__name__}")
    if not isinstance(s, (int, float)) or isinstance(s, bool):
        raise TypeError(f"s 必须是整数或浮点数，收到 {type(s).__name__}")
    if not all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in t):
        raise TypeError("t 的元素必须都是整数或浮点数")
    return tuple(x * s for x in t)

In [14]:
print("vec_scalar_mult((1, 2, 3), 2)   =", vec_scalar_mult((1, 2, 3), 2))
print("vec_scalar_mult((1, 2, 3), 0.5) =", vec_scalar_mult((1, 2, 3), 0.5))
print("vec_scalar_mult((1.5, 2.0), -2) =", vec_scalar_mult((1.5, 2.0), -2))
print("vec_scalar_mult((), 3)          =", vec_scalar_mult((), 3))

for t, s in [([1, 2], 2), ((1, "a"), 2), ((1, 2), "2"), ("12", 2)]:
    try:
        vec_scalar_mult(t, s)
    except TypeError as e:
        print(f"{t!r} * {s!r} -> TypeError: {e}")

vec_scalar_mult((1, 2, 3), 2)   = (2, 4, 6)
vec_scalar_mult((1, 2, 3), 0.5) = (0.5, 1.0, 1.5)
vec_scalar_mult((1.5, 2.0), -2) = (-3.0, -4.0)
vec_scalar_mult((), 3)          = ()
[1, 2] * 2 -> TypeError: t 必须是元组，收到 list
(1, 'a') * 2 -> TypeError: t 的元素必须都是整数或浮点数
(1, 2) * '2' -> TypeError: s 必须是整数或浮点数，收到 str
'12' * 2 -> TypeError: t 必须是元组，收到 str


### 3.2 `vec_inner_product`

两个参数都必须是元组（否则 `TypeError`），长度必须一致（否则 `ValueError`）；结果用 `float()` 转成浮点数。

In [15]:
def vec_inner_product(t1, t2):
    if not isinstance(t1, tuple) or not isinstance(t2, tuple):
        raise TypeError("两个参数都必须是元组")
    for name, t in (("t1", t1), ("t2", t2)):
        if not all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in t):
            raise TypeError(f"{name} 的元素必须都是整数或浮点数")
    if len(t1) != len(t2):
        raise ValueError(f"两个向量长度不一致：{len(t1)} vs {len(t2)}")
    return float(sum(a * b for a, b in zip(t1, t2)))

In [16]:
print("(1, 2, 3)·(4, 5, 6) =", vec_inner_product((1, 2, 3), (4, 5, 6)))
print("(0.5, 2)·(4, -1)    =", vec_inner_product((0.5, 2.0), (4.0, -1.0)))

try:
    vec_inner_product([1, 2], (1, 2))
except TypeError as e:
    print("列表输入   -> TypeError:", e)

try:
    vec_inner_product((1, 2), (1, 2, 3))
except ValueError as e:
    print("长度不一致 -> ValueError:", e)

(1, 2, 3)·(4, 5, 6) = 32.0
(0.5, 2)·(4, -1)    = 0.0
列表输入   -> TypeError: 两个参数都必须是元组
长度不一致 -> ValueError: 两个向量长度不一致：2 vs 3


### 3.3 `check_valid_mx`

按题目三条规则判断：外层是元组且每行是元组、所有行等长、元素都是整数或浮点数。空矩阵 `()` 三条规则都“空真”，因此视为有效，维度是否匹配留给 `mx_vec_mult` 检查。

In [ ]:
def check_valid_mx(mx):
    if not isinstance(mx, tuple):
        return False
    if not all(isinstance(row, tuple) for row in mx):
        return False
    if not mx:
        return True
    n = len(mx[0])
    for row in mx:
        if len(row) != n:
            return False
        if not all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in row):
            return False
    return True

In [20]:
matrices = [
    ((1, 2), (3, 4)),
    ((1.0, 2.5, -1.0), (0.0, 3.0, 4.0)),
    ((1, 2), (3,)),
    ([1, 2], [3, 4]),
    ((1, 2), (3, "4")),
    "not a matrix",
    (),
]

for mx in matrices:
    print(f"{mx!r:40s} -> {check_valid_mx(mx)}")

((1, 2), (3, 4))                         -> True
((1.0, 2.5, -1.0), (0.0, 3.0, 4.0))      -> True
((1, 2), (3,))                           -> False
([1, 2], [3, 4])                         -> False
((1, 2), (3, '4'))                       -> False
'not a matrix'                           -> False
()                                       -> True


### 3.4 `mx_vec_mult`

矩阵的每一行与向量做内积，得到长度为 $m$ 的元组；要求向量长度等于列数 $n$。

In [21]:
def mx_vec_mult(mx, vec):
    """返回矩阵 mx 与列向量 vec 相乘得到的数值元组。"""
    if not check_valid_mx(mx):
        raise TypeError("mx 不是有效矩阵（应是元素为数值的等长元组的元组）")
    if not isinstance(vec, tuple):
        raise TypeError(f"vec 必须是元组，收到 {type(vec).__name__}")
    if not all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in vec):
        raise TypeError("vec 的元素必须都是整数或浮点数")
    n = len(mx[0]) if mx else 0
    if len(vec) != n:
        raise ValueError(f"向量长度 {len(vec)} 与矩阵列数 {n} 不一致")
    return tuple(sum(row[j] * vec[j] for j in range(n)) for row in mx)

In [22]:
my_mx = ((1, 2), (3, 4), (5, 6))
print("矩阵：", my_mx)
print("向量 (2, 3) 与矩阵相乘：", mx_vec_mult(my_mx, (2, 3)))

try:
    mx_vec_mult([[1, 2], [3, 4]], (1, 2))
except TypeError as e:
    print("列表矩阵   -> TypeError:", e)

try:
    mx_vec_mult(((1, 2), (3, 4)), (1, 2, 3))
except ValueError as e:
    print("长度不匹配 -> ValueError:", e)

try:
    mx_vec_mult(((1, 2), (3, 4)), [1, 2])
except TypeError as e:
    print("列表向量   -> TypeError:", e)

矩阵： ((1, 2), (3, 4), (5, 6))
向量 (2, 3) 与矩阵相乘： (8, 18, 28)
列表矩阵   -> TypeError: mx 不是有效矩阵（应是元素为数值的等长元组的元组）
长度不匹配 -> ValueError: 向量长度 3 与矩阵列数 2 不一致
列表向量   -> TypeError: vec 必须是元组，收到 list


---

# 4. 更多向量的乐趣（面向对象）

### 4.1 `Vector` 类

`Vector(dim)` 只给维度时默认生成 `dim` 维零向量；元素在内部保存成元组，外部拿到的 `v.elements` 不会被原地修改。

In [23]:
class Vector:

    def __init__(self, dim, elements=None):
        if not isinstance(dim, int) or isinstance(dim, bool):
            raise TypeError(f"维度必须是整数，收到 {type(dim).__name__}")
        if dim < 0:
            raise ValueError(f"维度不能为负数：{dim}")
        if elements is None:
            elements = (0,) * dim            # 只给维度：默认零向量
        if not isinstance(elements, (list, tuple)):
            raise TypeError("元素必须是列表或元组")
        if len(elements) != dim:
            raise ValueError(f"维度 {dim} 与元素个数 {len(elements)} 不一致")
        for x in elements:
            if not isinstance(x, (int, float)) or isinstance(x, bool):
                raise TypeError(f"元素必须是整数或浮点数，收到 {type(x).__name__}")
        self.dim = dim
        self.elements = tuple(elements)      # 存成元组，避免被外部改动

    def __repr__(self):
        return f"Vector({self.dim}, {self.elements})"

    def _check_other(self, other, action):
        if not isinstance(other, Vector):
            raise TypeError(f"{action}的参数必须是 Vector")
        if self.dim != other.dim:
            raise ValueError(f"维度不同，无法{action}：{self.dim} vs {other.dim}")

    # ---------- 比较：坐标相同即相等，否则按元组字典序 ----------
    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_other(other, "比较")
        return self.elements == other.elements

    def __ne__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_other(other, "比较")
        return self.elements != other.elements

    def __lt__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_other(other, "比较")
        return self.elements < other.elements

    def __le__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_other(other, "比较")
        return self.elements <= other.elements

    def __gt__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_other(other, "比较")
        return self.elements > other.elements

    def __ge__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_other(other, "比较")
        return self.elements >= other.elements

    # ---------- 内积 ----------
    def dot(self, other):
        """返回两个向量的内积（浮点数）。"""
        self._check_other(other, "计算内积")
        return float(sum(a * b for a, b in zip(self.elements, other.elements)))

    # ---------- 乘法：标量数乘 / 逐元素相乘 ----------
    def __mul__(self, other):
        if isinstance(other, Vector):
            self._check_other(other, "逐元素相乘")
            return Vector(self.dim, [a * b for a, b in zip(self.elements, other.elements)])
        if not isinstance(other, (int, float)) or isinstance(other, bool):
            return NotImplemented
        return Vector(self.dim, [a * other for a in self.elements])

    def __rmul__(self, other):              # 支持 2 * v
        return self.__mul__(other)

In [24]:
v = Vector(3, [1, 2.5, -4])
print("v =", v, "| dim =", v.dim, "| elements =", v.elements)
print("默认零向量 Vector(4) =", Vector(4))

for args in [(-1,), (2.5,), (3, (1, 2)), (2, (1, "a"))]:
    try:
        Vector(*args)
    except (TypeError, ValueError) as e:
        print(f"Vector{args} -> {type(e).__name__}: {e}")

v = Vector(3, (1, 2.5, -4)) | dim = 3 | elements = (1, 2.5, -4)
默认零向量 Vector(4) = Vector(4, (0, 0, 0, 0))
Vector(-1,) -> ValueError: 维度不能为负数：-1
Vector(2.5,) -> TypeError: 维度必须是整数，收到 float
Vector(3, (1, 2)) -> ValueError: 维度 3 与元素个数 2 不一致
Vector(2, (1, 'a')) -> TypeError: 元素必须是整数或浮点数，收到 str


### 4.2 维度与元素：类属性还是实例属性？

**实例属性**，在 `__init__` 中用 `self.dim`、`self.elements` 赋值。理由：

1. 每个向量的维度和坐标都是**各自独立**的数据，而类属性由所有实例共享，`Vector(2, (1, 2))` 与 `Vector(3, (1, 2, 3))` 的值会互相覆盖；
2. 类属性只适合所有实例共享的量（常量、默认参数等），`dim`/`elements` 每个对象都不同；
3. 实例属性只属于当前对象，修改一个向量不会影响其他向量，符合封装的要求。

### 4.3 比较运算符

维度相同时按元组字典序比较（先比第 1 个坐标，再比第 2 个……），坐标完全相同即相等；维度不同时抛出 `ValueError`。

In [25]:
a = Vector(2, (2, 4))
b = Vector(2, (2, 5))
c = Vector(2, (3, 0))

print("a =", a, "| b =", b, "| c =", c)
print("a == Vector(2, [2, 4])：", a == Vector(2, [2, 4]))
print("a != b：", a != b)
print("a <  b：", a < b)
print("a <= b：", a <= b)
print("b >  a：", b > a)
print("b >= c：", b >= c)

try:
    a < Vector(3, (2, 4, 5))
except ValueError as e:
    print("维度不同 -> ValueError:", e)

a = Vector(2, (2, 4)) | b = Vector(2, (2, 5)) | c = Vector(2, (3, 0))
a == Vector(2, [2, 4])： True
a != b： True
a <  b： True
a <= b： True
b >  a： True
b >= c： False
维度不同 -> ValueError: 维度不同，无法比较：2 vs 3


### 4.4 `Vector.dot`

参数必须是 `Vector`（否则 `TypeError`），维度必须一致（否则 `ValueError`），返回 `float` 内积。

In [26]:
print("(1, 2)·(3, 4) =", Vector(2, (1, 2)).dot(Vector(2, (3, 4))))

try:
    Vector(2, (1, 2)).dot((3, 4))
except TypeError as e:
    print("非 Vector -> TypeError:", e)

try:
    Vector(2, (1, 2)).dot(Vector(3, (1, 2, 3)))
except ValueError as e:
    print("维度不同 -> ValueError:", e)

(1, 2)·(3, 4) = 11.0
非 Vector -> TypeError: 计算内积的参数必须是 Vector
维度不同 -> ValueError: 维度不同，无法计算内积：2 vs 3


### 4.5 乘法运算符

`__mul__` 同时处理“标量”与“向量”两种右操作数，`__rmul__` 让 `2 * v` 也能工作；两个向量相乘时逐元素相乘，结果仍是新的 `Vector`。

In [27]:
v = Vector(2, (2, 3))

print("2 * v              =", 2 * v)
print("v * 2              =", v * 2)
print("v * v              =", v * v)
print("v * Vector(2,(4,5))=", v * Vector(2, (4, 5)))

try:
    v * Vector(3, (1, 2, 3))
except ValueError as e:
    print("维度不同 -> ValueError:", e)

2 * v              = Vector(2, (4, 6))
v * 2              = Vector(2, (4, 6))
v * v              = Vector(2, (4, 9))
v * Vector(2,(4,5))= Vector(2, (8, 15))
维度不同 -> ValueError: 维度不同，无法逐元素相乘：2 vs 3
